# DeepFashion In-Shop — YOLOv8 Clothing Detection

Detects 3 clothing classes from DeepFashion In-shop bounding box annotations:

| YOLO class | Label     | DeepFashion cloth_type |
|:----------:|:---------:|:----------------------:|
| 0          | upper     | 1                      |
| 1          | lower     | 2                      |
| 2          | fullbody  | 3                      |

**Pipeline:** Dataset prep → YOLOv8m finetune → Inference (best box per class)

---
## Phase 1 — Dataset Preparation

In [1]:
# CELL 1 — Install Ultralytics
!pip install ultralytics -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.6 MB/s eta 0:00:00


In [2]:
# CELL 2 — Imports
import os
import shutil
from pathlib import Path
from collections import defaultdict

import pandas as pd
from PIL import Image
from tqdm import tqdm

In [3]:
# CELL 3 — Dataset Paths
# Adjust DATASET_ROOT if your mount point differs
DATASET_ROOT = "/kaggle/input/datasets/sartma/deepfashion-inshop" # Note: Double check if Kaggle includes 'datasets' in the path, usually it's just /kaggle/input/<slug>

BBOX_FILE      = os.path.join(DATASET_ROOT, "Anno/list_bbox_inshop.txt")
PARTITION_FILE = os.path.join(DATASET_ROOT, "Eval/list_eval_partition.txt")

# Just one "img" here. It will merge perfectly with the "img/MEN..." from the text file.
IMG_ROOT       = os.path.join(DATASET_ROOT, "img") 

print("BBOX file exists:      ", os.path.exists(BBOX_FILE))
print("Partition file exists:", os.path.exists(PARTITION_FILE))
print("Img root exists:       ", os.path.exists(IMG_ROOT))

BBOX file exists:       True
Partition file exists: True
Img root exists:        True


In [4]:
# CELL 4 — Create YOLO Directory Structure
YOLO_ROOT = "/content/deepfashion_yolo"

for split in ["train", "val"]:
    os.makedirs(os.path.join(YOLO_ROOT, f"images/{split}"), exist_ok=True)
    os.makedirs(os.path.join(YOLO_ROOT, f"labels/{split}"), exist_ok=True)

print("YOLO directory structure created at:", YOLO_ROOT)

YOLO directory structure created at: /content/deepfashion_yolo


In [5]:
# CELL 5 — Read Partition File
# Split column values: 'train', 'query', 'gallery'
# We treat query + gallery as val
partition_df = pd.read_csv(
    PARTITION_FILE,
    skiprows=2,
    sep=r"\s+",
    names=["image_name", "item_id", "split"]
)

partition_map = dict(zip(partition_df.image_name, partition_df.split))

print(f"Total partition entries: {len(partition_map)}")
print(partition_df["split"].value_counts())

Total partition entries: 52712
split
train      25882
query      14218
gallery    12612
Name: count, dtype: int64


In [6]:
# CELL 6 — Read Bounding Box Annotations
bbox_df = pd.read_csv(
    BBOX_FILE,
    skiprows=2,
    sep=r"\s+",
    names=["image_name", "cloth_type", "pose_type", "x1", "y1", "x2", "y2"]
)

print(f"Total bbox entries: {len(bbox_df)}")
print("cloth_type distribution:")
print(bbox_df["cloth_type"].value_counts())
bbox_df.head()

Total bbox entries: 52712
cloth_type distribution:
cloth_type
1    33487
2    10494
3     8731
Name: count, dtype: int64


,image_name,cloth_type,pose_type,x1,y1,x2,y2
0,img/WOMEN/Blouses_Shirts/id_00000001/02_1_fron...,1,1,50,49,208,235
1,img/WOMEN/Blouses_Shirts/id_00000001/02_2_side...,1,2,119,48,136,234
2,img/WOMEN/Blouses_Shirts/id_00000001/02_3_back...,1,3,50,42,213,240
3,img/WOMEN/Blouses_Shirts/id_00000001/02_4_full...,1,4,82,30,162,129
4,img/WOMEN/Dresses/id_00000002/02_1_front.jpg,3,1,65,45,233,252


In [7]:
# CELL 7 — Convert Annotations to YOLO Format
#
# DeepFashion bbox: absolute pixel coords (x1, y1, x2, y2)
# YOLO format:      normalized (x_center, y_center, width, height) in [0,1]
#
# Class remapping:
#   cloth_type 1 (upper)    → YOLO class 0
#   cloth_type 2 (lower)    → YOLO class 1
#   cloth_type 3 (fullbody) → YOLO class 2

annotations = defaultdict(list)
skipped = 0

for _, row in tqdm(bbox_df.iterrows(), total=len(bbox_df), desc="Converting"):

    image_rel  = row.image_name
    image_path = os.path.join(IMG_ROOT, image_rel)

    if not os.path.exists(image_path):
        skipped += 1
        continue

    # Determine split: only 'train' → train; everything else → val
    raw_split  = partition_map.get(image_rel, "train")
    split_name = "train" if raw_split == "train" else "val"

    # Get image dimensions (needed for normalization)
    img     = Image.open(image_path)
    w, h    = img.size

    x1, y1, x2, y2 = row.x1, row.y1, row.x2, row.y2

    # Guard against degenerate boxes
    if x2 <= x1 or y2 <= y1:
        skipped += 1
        continue

    xc = ((x1 + x2) / 2) / w
    yc = ((y1 + y2) / 2) / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h

    cls = int(row.cloth_type) - 1  # 1→0, 2→1, 3→2

    annotations[(split_name, image_rel)].append(
        f"{cls} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}"
    )

print(f"\nAnnotations built: {len(annotations)} images")
print(f"Skipped (missing / bad box): {skipped}")

Converting: 100%|██████████| 52712/52712 [09:15<00:00, 94.84it/s]


Annotations built: 52712 images
Skipped (missing / bad box): 0


In [8]:
# CELL 8 — Copy Images and Write Label Files
#
# NOTE: filenames inside the dataset can collide across categories.
# We preserve the subdirectory structure in the filename to avoid this:
#   img/MEN/Denim/id_00000080/01_1_front.jpg
#   → MEN_Denim_id_00000080_01_1_front.jpg

def safe_name(image_rel: str) -> str:
    """Flatten path separators to underscores for a flat YOLO images/ dir."""
    parts = Path(image_rel).parts
    return "_".join(parts)


for (split_name, image_rel), labels in tqdm(
    annotations.items(), desc="Saving"
):
    src_img  = os.path.join(IMG_ROOT, image_rel)
    flat     = safe_name(image_rel)          # e.g. img_MEN_Denim_..._front.jpg
    stem     = Path(flat).stem              # strip .jpg

    dst_img   = os.path.join(YOLO_ROOT, f"images/{split_name}/{flat}")
    dst_label = os.path.join(YOLO_ROOT, f"labels/{split_name}/{stem}.txt")

    shutil.copy(src_img, dst_img)

    with open(dst_label, "w") as f:
        f.write("\n".join(labels))

print("Done saving images + labels.")

Saving: 100%|██████████| 52712/52712 [02:02<00:00, 429.18it/s]

Done saving images + labels.


In [9]:
# CELL 9 — Write data.yaml
yaml_text = """\
path: /content/deepfashion_yolo

train: images/train
val:   images/val

nc: 3
names:
  0: upper
  1: lower
  2: fullbody
"""

yaml_path = os.path.join(YOLO_ROOT, "data.yaml")
with open(yaml_path, "w") as f:
    f.write(yaml_text)

print("data.yaml written to:", yaml_path)
print(yaml_text)

data.yaml written to: /content/deepfashion_yolo/data.yaml
path: /content/deepfashion_yolo

train: images/train
val:   images/val

nc: 3
names:
  0: upper
  1: lower
  2: fullbody



In [10]:
# CELL 10 — Verify Dataset Counts
import glob

for split in ["train", "val"]:
    n_imgs   = len(glob.glob(f"{YOLO_ROOT}/images/{split}/*.jpg"))
    n_labels = len(glob.glob(f"{YOLO_ROOT}/labels/{split}/*.txt"))
    print(f"{split}: {n_imgs} images, {n_labels} labels", end="")
    if n_imgs != n_labels:
        print("  ⚠️  MISMATCH — check for missing files")
    else:
        print("  ✓")

train: 25882 images, 25882 labels  ✓
val: 26830 images, 26830 labels  ✓


---
## Phase 2 — YOLOv8m Finetuning

Key choices:
- **Model**: `yolov8m.pt` (pretrained COCO) — good balance of speed and accuracy on T4
- **imgsz 640** — captures clothing detail
- **AdamW + cosine LR** — better finetuning convergence
- **close_mosaic=10** — disables mosaic for the final 10 epochs, improving localization
- **mixup + mosaic** — generalization without destroying garment structure

In [11]:
# CELL 11 — Load YOLOv8m
from ultralytics import YOLO

model = YOLO("yolov8m.pt")  # downloads pretrained COCO weights if not cached
print(model.info())

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLOv8m summary: 169 layers, 25,902,640 parameters, 0 gradients, 79.3 GFLOPs
(169, 25902640, 0, 79.3204224)


In [12]:
# CELL 12 — Train
results = model.train(
    data   = "/content/deepfashion_yolo/data.yaml",

    # ── Duration ──────────────────────────────────────────
    epochs       = 100,
    patience     = 25,        # early-stop if no improvement for 25 epochs

    # ── Resolution ────────────────────────────────────────
    imgsz        = 256,

    # ── Batch / Workers ───────────────────────────────────
    batch        = 16,
    workers      = 4,

    # ── Optimiser ─────────────────────────────────────────
    optimizer    = "AdamW",
    lr0          = 1e-3,
    lrf          = 1e-5,      # final LR = lr0 * lrf
    weight_decay = 0.0005,
    warmup_epochs= 5,
    cos_lr       = True,      # cosine annealing

    # ── Augmentation ──────────────────────────────────────
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    degrees      = 5,
    translate    = 0.1,
    scale        = 0.4,
    fliplr       = 0.5,
    mosaic       = 1.0,
    mixup        = 0.15,
    close_mosaic = 10,        # disable mosaic in last 10 epochs

    # ── Misc ──────────────────────────────────────────────
    pretrained   = True,
    amp          = True,      # mixed precision
    device       = 0,

    project      = "deepfashion_yolo",
    name         = "yolov8m_finetune"
)

print("Training complete. Best weights saved at:",
      results.save_dir + "/weights/best.pt")

Ultralytics 8.4.48 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/deepfashion_yolo/data.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=256, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=1e-05, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_finetune, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=Tru

TypeError: unsupported operand type(s) for +: 'PosixPath' and 'str'

---
## Phase 3 — Validation & Inference

In [ ]:
# CELL 13 — Evaluate on Validation Set
# Load best checkpoint
best_model = YOLO("deepfashion_yolo/yolov8m_finetune/weights/best.pt")

metrics = best_model.val(
    data   = "/content/deepfashion_yolo/data.yaml",
    imgsz  = 640,
    batch  = 16,
    device = 0
)

print(f"mAP@50:      {metrics.box.map50:.4f}")
print(f"mAP@50-95:   {metrics.box.map:.4f}")
print(f"Per-class AP@50: {metrics.box.ap50}")

In [ ]:
# # CELL 14 — Single-image Inference
# # Change TEST_IMAGE to your actual test image path
# TEST_IMAGE = "/content/test.jpg"

# results = best_model(
#     TEST_IMAGE,
#     imgsz = 640,
#     conf  = 0.35,   # filter weak detections
#     iou   = 0.45
# )

# # Show inline
# results[0].show()

In [ ]:
# CELL 15 — Keep Best Box Per Class (With Cross-Class Overlap Filtering)

CLASS_NAMES = {0: "upper", 1: "lower", 2: "fullbody"}
CONF_THRESH  = 0.35
IOU_THRESH   = 0.60  # If two different classes overlap by more than 50%, drop the weaker one

def calculate_iou(boxA, boxB):
    """Calculates Intersection over Union for two [x1, y1, x2, y2] boxes."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    if interArea == 0:
        return 0.0

    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    return interArea / float(boxAArea + boxBArea - interArea)


def best_boxes_per_class(result, conf_thresh=CONF_THRESH, iou_thresh=IOU_THRESH):
    """
    Returns exactly ONE box per class. 
    Filters out lower-confidence boxes that heavily overlap with a different class.
    """
    # 1. Get the absolute best box for each class (your original logic)
    best_candidates = {}
    for box in result.boxes:
        conf = float(box.conf[0])
        if conf < conf_thresh:
            continue
            
        cls = int(box.cls[0])
        if cls not in best_candidates or conf > float(best_candidates[cls].conf[0]):
            best_candidates[cls] = box

    # 2. Cross-Class NMS: Compare the winners against each other
    final_boxes = {}
    
    for cls_id, box in best_candidates.items():
        keep = True
        box_coords = box.xyxy[0].tolist()
        box_conf = float(box.conf[0])
        
        for other_cls_id, other_box in best_candidates.items():
            if cls_id == other_cls_id:
                continue
                
            other_coords = other_box.xyxy[0].tolist()
            other_conf = float(other_box.conf[0])
            
            # If there is heavy physical overlap with another class...
            if calculate_iou(box_coords, other_coords) > iou_thresh:
                # ...and the other class has a higher confidence, drop this one
                if other_conf > box_conf:
                    keep = False
                    break
                    
        if keep:
            final_boxes[cls_id] = box
            
    return final_boxes


# --- Run it ---
# best = best_boxes_per_class(results[0])

# print(f"Detected {len(best)} distinct clothing region(s):\n")
# for cls_id, box in sorted(best.items()):
#     xyxy = [round(v, 1) for v in box.xyxy[0].tolist()]
#     conf = float(box.conf[0])
#     print(f"  [{CLASS_NAMES[cls_id]:>8}]  conf={conf:.3f}  box={xyxy}")

In [ ]:
# CELL 16 — Batch Inference Helper
#
# Use this when you want to run predictions across a folder of images.

from pathlib import Path


def predict_folder(model, folder: str, conf_thresh: float = 0.35):
    """
    Run best-box-per-class inference on every .jpg in `folder`.
    Returns a list of dicts: {file, upper, lower, fullbody}
    """
    image_paths = sorted(Path(folder).glob("*.jpg"))
    records = []

    for img_path in tqdm(image_paths, desc="Predicting"):
        res  = model(str(img_path), imgsz=640, conf=conf_thresh, verbose=False)
        best = best_boxes_per_class(res[0], conf_thresh)

        row = {"file": img_path.name}
        for cid, cname in CLASS_NAMES.items():
            if cid in best:
                box  = best[cid]
                xyxy = box.xyxy[0].tolist()
                row[cname] = {
                    "conf": round(float(box.conf[0]), 4),
                    "bbox": [round(v, 1) for v in xyxy]
                }
            else:
                row[cname] = None
        records.append(row)

    return records


# Example usage:
# preds = predict_folder(best_model, "/content/my_test_images")
# preds[0]
print("predict_folder() helper defined. Call it on any folder of .jpg images.")

---
## Notes & Next Steps

### Expected Performance
With 3 classes and 50k+ training images you should see:
- **mAP@50 ≥ 85%** with the settings above
- **mAP@50 ≥ 90%** possible with longer training (120 epochs) or `yolov8l`

### Scaling Up
| Change | Effect |
|:-------|:-------|
| Switch to `yolov8l.pt` | +1–2 mAP, ~1.5× slower |
| `imgsz=768` | Better for fine-grained details, needs more VRAM |
| `epochs=120` | Useful if validation mAP is still rising at epoch 100 |

### Real-World Robustness
DeepFashion images are studio-centered shots. For street/in-the-wild performance, add:
- DeepFashion2
- ModaNet
- Street fashion images

### Recommended Future Upgrade
After detection is solid, add a **YOLOv8-seg** head trained on `Anno/segmentation/` for precise clothing masks — especially helpful for overlapping garments.